# Récord de puntos por jugador

Sistemas de Big Data 5074 · UT1.2 · Diapositivas 17–22.

Versión para completar. Puedes trabajar en este cuaderno en lugar de `alumno.py`. Sustituye los `NotImplementedError` por tu código. Las celdas con tareas pendientes se detendrán hasta que las completes.

Solo biblioteca estándar de Python 3.10 o posterior. Los datos están incluidos. No requiere instalaciones ni archivos externos. Puedes subir este `.ipynb` a Colab. Los ejemplos MapReduce se simulan localmente y en serie, sin ejecutar Hadoop.


# Ejemplo 3. Récord por jugador

Vas a completar el ejemplo de las diapositivas 17–22 de `Evaluacion_UT1_2.pptx`, de UT1.2 de SBD (5074). El objetivo es obtener la puntuación máxima de **cada jugador** entre varios archivos de partidas.

Abre `alumno.py` y completa sus cinco TODO. Los bloques `# %%` permiten avanzar por etapas en un editor que reconozca celdas. Con Python 3.10 o posterior también puedes ejecutar el archivo completo:

```console
python3 alumno.py
```

Al principio se detendrá con `NotImplementedError`. Sustituye cada instrucción de ese tipo por el código que pide su comentario. Los imports, los datos y la visualización final ya están preparados. Solo necesitas la biblioteca estándar.

## Datos de la práctica

Cada línea contiene `fecha jugador puntos`. Las tres primeras líneas de mayo reproducen el texto visible de la diapositiva 19. La cuarta línea de mayo y las tres de junio amplían los datos para obtener las listas de la diapositiva 22. **Las fechas de esos cuatro registros añadidos son ficticias.**

El diccionario `FICHEROS` contiene todos los datos. Los archivos de `datos/` son una copia para consultarlos. El programa utiliza el diccionario y puede ejecutarse desde cualquier carpeta sin leer archivos externos. Ambos meses están en UTF-8, sin BOM y con saltos de línea LF.

## Bloque de lectura: TODO 1

Completa `leer_registros(texto)`. Devuelve una lista de pares `(offset, linea)`. Usa `BytesIO`, consulta `tell()` **antes** de cada `readline()` y decodifica cada línea con UTF-8. Retira el terminador de línea al construir el texto del registro.

El offset indica cuántos bytes hay antes del comienzo de la línea dentro de ese archivo. Calcula los offsets de mayo a partir de la lectura binaria. La `Í` ocupa dos bytes en UTF-8. Si el texto usa CRLF, cuenta los dos bytes del salto. La última línea también se procesa cuando no termina en LF.

Cada archivo empieza en el byte 0. Este valor no representa la fecha ni la posición en una lista, y tampoco identifica un registro en toda la colección. La diapositiva 21 imprime `0, 23, 48`, pero esos números no se reproducen con el texto visible normalizado a UTF-8 y LF que usamos aquí.

## Bloque Map: TODO 2

Completa `mapa_registro(offset, linea)`. Extrae la puntuación desde la derecha con `rsplit(" ", 1)` y separa la fecha del jugador con `split(maxsplit=1)`. Así el nombre puede contener espacios. Devuelve, por ejemplo, `("MARÍA", 18)`.

La fecha debe ser válida y los puntos deben ser enteros. Si el registro está vacío o tiene un formato incorrecto, señala un `ValueError` con la línea y su offset. Las puntuaciones negativas sí se aceptan.

## Bloque Shuffle: TODO 3

Completa `agrupar(pares)`. Usa el jugador como clave de un diccionario y añade sus puntuaciones a una lista. Construye los grupos con los siete registros y comprueba que cada puntuación llega al jugador correspondiente.

## Bloque Reduce: TODO 4

Completa `reducir(jugador, puntos)`. Recorre los puntos para calcular el máximo y devuelve `(jugador, maximo)`. Empieza por la primera puntuación para que también funcione si todas son negativas. Una lista vacía debe provocar `ValueError`.

En un empate devolvemos la puntuación máxima una sola vez. El resultado no incluye fechas ni partidas ganadoras.

## Bloque de ejecución: TODO 5

Completa la coordinación dentro de `ejecutar_mapreduce(ficheros)`. Recorre todos los archivos, lee sus registros y aplica Map. Mantén una emisión por registro. Cuando haya un error, añade el nombre del archivo y vuelve a lanzarlo.

Después conecta Shuffle y Reduce: agrupa las emisiones, recorre los grupos y guarda el máximo de cada jugador en `resultados`. Para seguir el proceso en clase, puedes mostrar las emisiones y los grupos dentro de esta función.

Cuando termines, ejecuta el programa. El bloque final llama a tus funciones para mostrar la lectura y comparar los récords calculados con los tres registros originales y con los siete de la práctica. Revisa manualmente qué partidas explican los resultados.

Cada etiqueta de jugador se considera una misma identidad en todos los archivos. `MARÍA` y `María` formarían claves diferentes. Estamos calculando un récord por jugador. Esta simulación trabaja en serie en tu ordenador y guarda los grupos en memoria. Hadoop distribuye el trabajo y gestiona aspectos que aquí no implementamos.



## Código 1 Datos de la práctica


In [1]:
"""Ejemplo 3: completa las cinco partes indicadas con TODO.

Python 3.10 o posterior. No necesitas instalar paquetes ni abrir los .txt.
Las tres primeras líneas de mayo proceden de la diapositiva 19.
La cuarta línea de mayo y las fechas de junio son inventadas para practicar.
Sus puntuaciones completan las listas de la diapositiva 22.
"""

from datetime import datetime
from io import BytesIO


FICHEROS = {
    "Mayo_2020.txt": (
        "01-05-2020 MARÍA 18\n"
        "03-05-2020 JUAN 23\n"
        "07-05-2020 MARÍA 45\n"
        "12-05-2020 JUAN 5\n"
    ),
    "Junio_2020.txt": (
        "02-06-2020 MARÍA 30\n"
        "09-06-2020 MARÍA 60\n"
        "16-06-2020 JUAN 45\n"
    ),
}


## Código 2 Lectura de líneas y offsets en bytes


In [2]:
def leer_registros(texto: str) -> list[tuple[int, str]]:
    """Devuelve (offset en bytes, línea sin terminador) por cada línea."""
    flujo = BytesIO(texto.encode("utf-8"))
    registros = []
    while True:
        offset = flujo.tell()
        linea_bytes = flujo.readline()
        if not linea_bytes:
            break
        # Decodifica y elimina los caracteres de fin de línea (\r o \n)
        linea = linea_bytes.decode("utf-8").rstrip("\r\n")
        registros.append((offset, linea))
    return registros

## Código 3 Map: de una línea a un par jugador y puntos


In [3]:
def mapa_registro(offset: int, linea: str) -> tuple[str, int]:
    """Devuelve (jugador, puntos). Un registro inválido provoca ValueError."""
    try:
        partes = linea.strip().rsplit(" ", 1)
        if len(partes) != 2:
            raise ValueError()

        cuerpo, str_puntos = partes
        puntos = int(str_puntos)

        fecha, jugador = cuerpo.split(maxsplit=1)
        datetime.strptime(fecha, "%d-%m-%Y")  # Valida que la fecha sea correcta

        return (jugador, puntos)
    except Exception as e:
        raise ValueError(f"Error en offset {offset} con línea '{linea}': {e}")

## Código 4 Shuffle: reunir los puntos de cada jugador


In [4]:
def agrupar(pares: list[tuple[str, int]]) -> dict[str, list[int]]:
    """Devuelve un diccionario de jugador a lista de puntuaciones."""
    grupos = {}
    for jugador, puntos in pares:
        if jugador not in grupos:
            grupos[jugador] = []
        grupos[jugador].append(puntos)
    return grupos

## Código 5 Reduce: máximo de los puntos de un jugador


In [5]:
def reducir(jugador: str, puntos: list[int]) -> tuple[str, int]:
    """Devuelve (jugador, máximo). Una lista vacía provoca ValueError."""
    if not puntos:
        raise ValueError(f"La lista de puntos para el jugador {jugador} está vacía.")

    maximo = puntos[0]
    for p in puntos[1:]:
        if p > maximo:
            maximo = p

    return (jugador, maximo)

## Código 6 Ejecución de MapReduce para una colección de archivos


In [6]:
def ejecutar_mapreduce(ficheros: dict[str, str]) -> dict[str, int]:
    """Conecta las etapas de lectura, Map, Shuffle y Reduce."""
    emisiones = []
    resultados = {}

    for nombre_archivo, texto in ficheros.items():
        try:
            registros = leer_registros(texto)
            for offset, linea in registros:
                par = mapa_registro(offset, linea)
                emisiones.append(par)
        except Exception as e:
            raise ValueError(f"Error procesando el archivo '{nombre_archivo}': {e}")

    # Opcional para seguir el proceso en clase:
    # print("Emisiones:", emisiones)

    grupos = agrupar(emisiones)

    # Opcional para seguir el proceso en clase:
    # print("Grupos:", grupos)

    for jugador, puntos_lista in grupos.items():
        _, maximo = reducir(jugador, puntos_lista)
        resultados[jugador] = maximo

    return resultados

## Código 7 Observación del proceso y de los resultados


In [7]:
if __name__ == "__main__":
    print("Offsets y líneas de Mayo_2020.txt (UTF-8, LF):")
    print(leer_registros(FICHEROS["Mayo_2020.txt"]))

    tres_lineas = FICHEROS["Mayo_2020.txt"].splitlines(keepends=True)[:3]
    originales = {"Mayo_2020.txt": "".join(tres_lineas)}
    print("\nRécords con los tres registros originales:")
    print(ejecutar_mapreduce(originales))
    print("\nRécords con los siete registros de la práctica:")
    print(ejecutar_mapreduce(FICHEROS))

Offsets y líneas de Mayo_2020.txt (UTF-8, LF):
[(0, '01-05-2020 MARÍA 18'), (21, '03-05-2020 JUAN 23'), (40, '07-05-2020 MARÍA 45'), (61, '12-05-2020 JUAN 5')]

Récords con los tres registros originales:
{'MARÍA': 45, 'JUAN': 23}

Récords con los siete registros de la práctica:
{'MARÍA': 60, 'JUAN': 45}


## Explicación de tus resultados

Anota aquí tus conclusiones y las respuestas que pide el guion.





### 1. Análisis de los offsets y lectura binaria (TODO 1)
* **Comportamiento de los bytes:** Al leer el archivo `Mayo_2020.txt` mediante `BytesIO` y `readline()`, los desplazamientos (*offsets*) obtenidos son `[(0, '01-05-2020 MARÍA 18'), (21, '03-05-2020 JUAN 23'), (40, '07-05-2020 MARÍA 45'), (61, '12-05-2020 JUAN 5')]`.
* **Cálculo de los saltos:** Cada valor de offset indica la posición exacta en bytes donde comienza cada línea dentro del flujo binario. Estos valores reflejan la longitud en bytes de las líneas anteriores (incluyendo los caracteres de salto de línea `\n`). Por ejemplo, la letra con tilde `Í` en **MARÍA** ocupa 2 bytes en codificación UTF-8, lo cual se contabiliza correctamente en el desplazamiento acumulado del archivo.


### 2. Comparativa de récords (Originales vs. Práctica completa)
* **Con los 3 registros originales (Mayo):**
  * Partidas consideradas: `MARÍA (18)`, `JUAN (23)` y `MARÍA (45)`.
  * **Récord calculado:** `{'MARÍA': 45, 'JUAN': 23}`.
  * *Explicación:* MARÍA participa en dos ocasiones (con 18 y 45 puntos), siendo `45` su puntuación máxima. JUAN solo cuenta con una aparición de `23` puntos.
* **Con los 7 registros de la práctica (Mayo y Junio):**
  * Partidas añadidas: `JUAN (5)` en mayo; `MARÍA (30)`, `MARÍA (60)` y `JUAN (45)` en junio.
  * **Récord calculado:** `{'MARÍA': 60, 'JUAN': 45}`.
  * *Explicación:* Al integrar los datos de junio, las puntuaciones máximas de ambos jugadores aumentan gracias a sus partidas en dicho mes (`60` para MARÍA y `45` para JUAN), actualizando así el récord histórico por jugador entre todos los archivos procesados.


### 3. Consideraciones del modelo MapReduce simulado
* **Identidad de las claves:** Las etiquetas de los jugadores distinguen mayúsculas de minúsculas (por ejemplo, `MARÍA` y `María` se tratarían como entidades independientes).
* **Simulación vs. Entorno Distribuido:** Esta práctica se ejecuta de manera secuencial y local en memoria (usando estructuras nativas de Python como diccionarios y listas), permitiendo depurar fácilmente los errores de formato (`ValueError`) gracias al rastreo de los *offsets* y nombres de archivos. En un entorno real de **Hadoop**, este mismo flujo de trabajo (*Map, Shuffle y Reduce*) se distribuiría de forma masiva y paralela en un clúster de nodos.